In [6]:
import yfinance as yf
import pandas as pd
from ta.trend import EMAIndicator
from ta.momentum import RSIIndicator
from ta.trend import MACD

# Function to get Nifty 100 tickers from Wikipedia
def get_nifty100_tickers():
    url = 'https://en.wikipedia.org/wiki/NIFTY_50'
    tables = pd.read_html(url)
    df = tables[2]  # Assuming the table with symbols is the third table (index 2)
    tickers = df['Symbol'].tolist()
    return [ticker + '.NS' for ticker in tickers]

# Get the list of tickers
tickers = get_nifty100_tickers()

# Initialize an empty list to store results
all_signals = []

# Loop over each ticker
for ticker in tickers:
    try:
        # Fetch 1 year data
        data = yf.download(ticker, period='1y', progress=False)
        
        if data.empty:
            continue
        
        # Calculate EMA (e.g., 9 and 21)
        ema9 = EMAIndicator(close=data['Close'], window=9).ema_indicator()
        ema21 = EMAIndicator(close=data['Close'], window=21).ema_indicator()
        
        # Calculate RSI
        rsi = RSIIndicator(close=data['Close'], window=14).rsi()
        
        # Calculate MACD
        macd = MACD(close=data['Close'])
        macd_line = macd.macd()
        signal_line = macd.macd_signal()
        
        # Generate signals based on EMA crossover and RSI/MACD confirmation
        signals = []
        for i in range(len(data)):
            buy_signal = (ema9.iloc[i] > ema21.iloc[i]) and (rsi.iloc[i] < 70) and (macd_line.iloc[i] > signal_line.iloc[i])
            sell_signal = (ema9.iloc[i] < ema21.iloc[i]) and (rsi.iloc[i] > 30) and (macd_line.iloc[i] < signal_line.iloc[i])
            if buy_signal:
                signal = 'BUY'
            elif sell_signal:
                signal = 'SELL'
            else:
                signal = 'HOLD'
            signals.append(signal)
        
        # Add to data
        data['EMA9'] = ema9
        data['EMA21'] = ema21
        data['RSI'] = rsi
        data['MACD'] = macd_line
        data['Signal'] = signals
        data['Ticker'] = ticker
        
        # Append to all_signals
        all_signals.append(data.reset_index())
    
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")

# Combine all into one DataFrame
if all_signals:
    combined_df = pd.concat(all_signals, ignore_index=True)
    # Save to CSV
    combined_df.to_csv('nifty100_signals.csv', index=False)
    print("Signals saved to nifty100_signals.csv")
else:
    print("No data fetched.")

HTTPError: HTTP Error 403: Forbidden